# Extract India State Boundaries

**Purpose:** Extract and filter India states from Natural Earth Admin 1 shapefile

**Input:** Downloaded `ne_10m_admin_1_states_provinces.zip`

**Output:** `data/raw/geography/india_states.shp`

In [5]:
import geopandas as gpd
import zipfile
from pathlib import Path
import shutil

print("✓ Imports complete")

✓ Imports complete


In [6]:
# STEP 1: Set path to the already-extracted folder
extracted_folder = Path("~/Downloads/ne_10m_admin_1_states_provinces").expanduser()

# Check if folder exists
if not extracted_folder.exists():
    print(f"❌ Folder not found: {extracted_folder}")
    print("\nPlease update the path above to your extracted folder.")
    print("\nFolders in your Downloads:")
    downloads = Path.home() / "Downloads"
    for f in sorted(downloads.iterdir())[:15]:
        if f.is_dir():
            print(f"   📁 {f.name}")
else:
    print(f"✓ Found: {extracted_folder}")
    print("\nFiles in folder:")
    for f in sorted(extracted_folder.glob("*"))[:20]:
        print(f"   {f.name}")

✓ Found: /Users/a0488240/Downloads/ne_10m_admin_1_states_provinces

Files in folder:
   ne_10m_admin_1_states_provinces.README.html
   ne_10m_admin_1_states_provinces.VERSION.txt
   ne_10m_admin_1_states_provinces.cpg
   ne_10m_admin_1_states_provinces.dbf
   ne_10m_admin_1_states_provinces.prj
   ne_10m_admin_1_states_provinces.shp
   ne_10m_admin_1_states_provinces.shx


In [7]:
# STEP 2: Find the shapefile in the extracted folder
shapefiles = list(extracted_folder.glob("*.shp"))

if not shapefiles:
    print("❌ No .shp files found!")
    print(f"\nSearching in: {extracted_folder}")
else:
    shapefile = shapefiles[0]
    print(f"✓ Found shapefile: {shapefile.name}")
    print(f"\nAll shapefiles in folder:")
    for shp in shapefiles:
        print(f"   {shp.name}")

✓ Found shapefile: ne_10m_admin_1_states_provinces.shp

All shapefiles in folder:
   ne_10m_admin_1_states_provinces.shp


In [ ]:
# STEP 3: Read the shapefile using fiona directly to avoid compatibility issue
import fiona
from shapely.geometry import shape

print(f"Reading: {shapefile.name}...\n")

# Open with fiona directly
records = []
with fiona.open(str(shapefile), 'r') as src:
    crs = src.crs
    print(f"CRS: {crs}")
    
    # Read all records
    for feature in src:
        records.append({
            'geometry': shape(feature['geometry']),
            **feature['properties']
        })

# Create GeoDataFrame manually
all_states = gpd.GeoDataFrame(records, crs=crs)

print(f"\n✓ Loaded {len(all_states):,} global admin units")
print(f"\nColumns: {list(all_states.columns)}")
print(f"\nBounds: {all_states.total_bounds}")

In [9]:
# STEP 4: Preview the data
print("Sample records:")
all_states[["admin", "name", "type"]].head(10)

Sample records:


NameError: name 'all_states' is not defined

In [ ]:
# Check if India exists
print("Countries containing 'India':")
india_like = all_states[all_states["admin"].str.contains("India", case=False, na=False)]
print(india_like["admin"].unique())

In [ ]:
# STEP 5: Filter for India only
india_states = all_states[all_states["admin"] == "India"].copy()

print(f"✓ Found {len(india_states)} Indian states/territories\n")
print("Indian states:")
for idx, state in enumerate(india_states["name"].sort_values(), 1):
    print(f"   {idx:2d}. {state}")

In [ ]:
# STEP 6: Preview the map
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 12))
india_states.boundary.plot(ax=ax, linewidth=0.8, edgecolor="darkblue")
india_states.plot(
    ax=ax, color="lightblue", alpha=0.3, edgecolor="darkblue", linewidth=0.8
)

ax.set_title("India State Boundaries", fontsize=14, fontweight="bold")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Map bounds: {india_states.total_bounds}")
print(f"  [min_lon, min_lat, max_lon, max_lat]")

In [ ]:
# STEP 7: Save to project directory
current = Path.cwd()
project_root = current if (current / "CLAUDE.md").exists() else current.parent.parent

output_dir = project_root / "data" / "raw" / "geography"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "india_states.shp"
india_states.to_file(output_path)

print(f"✓ Saved to: {output_path}")
print(f"\nFiles created:")
for f in sorted(output_dir.glob("india_states.*")):
    print(f"   {f.name}")

In [ ]:
# STEP 8: No cleanup needed since we're reading from an existing folder
print("✓ No temporary files to clean up (using existing extracted folder)")

In [ ]:
# STEP 9: Verify you can load it back
# Use fiona.open() workaround to avoid compatibility issue
with fiona.open(str(output_path), 'r') as src:
    test_load_count = len(list(src))

print(f"✓ Verification: Loaded {test_load_count} states")
print(f"\n{'='*60}")
print("SUCCESS! India boundaries extracted and saved!")
print(f"\nTo use in your notebooks, load with fiona workaround:")
print(f"\n```python")
print(f"import geopandas as gpd")
print(f"import fiona")
print(f"from shapely.geometry import shape")
print(f"")
print(f"# Read using fiona directly (avoids compatibility issue)")
print(f"records = []")
print(f"with fiona.open('data/raw/geography/india_states.shp', 'r') as src:")
print(f"    crs = src.crs")
print(f"    for feature in src:")
print(f"        records.append({{'geometry': shape(feature['geometry']), **feature['properties']}})")
print(f"")
print(f"india_states = gpd.GeoDataFrame(records, crs=crs)")
print(f"```")
print(f"\nSaved to: {output_path}")